# Pipeline 1 — Donor Lapse Risk (Churn) Predictor

**Created:** 2026-04-06T17:23:54.994489Z

This notebook follows **CRISP-DM** while also satisfying the IS455 rubric sections:
- Problem Framing
- Data Acquisition, Preparation & Exploration
- Modeling & Feature Selection
- Evaluation & Interpretation
- Causal and Relationship Analysis
- Deployment Notes


## CRISP-DM Overview

### 1) Business Understanding
Goal: identify supporters likely to stop giving soon so leadership can intervene with targeted outreach.

### 2) Data Understanding
Use supporters + donation history up to a cutoff date; label donation activity in the next 90 days.

### 3) Data Preparation
Aggregate donation history to supporter-level features (RFM + mix features).

### 4) Modeling
Predictive: Gradient Boosting. Explanatory: Logistic Regression for interpretability.

### 5) Evaluation
Use ROC-AUC/F1 to handle imbalance. Discuss cost of false positives/negatives for fundraising operations.

### 6) Deployment
Export risk scores, import into `/api/ml/import`, display in `/app/ml` and (later) Donors page.


## 1) Problem Framing (Rubric)

State:
- the business question,
- who cares,
- why it matters,
- predictive vs explanatory goals.

We build **two models**:
- Predictive (optimize out-of-sample performance)
- Explanatory (interpretability / relationship analysis)


## 2) Data Acquisition, Preparation & Exploration (Rubric)

Rules to avoid leakage:
- Define an **as-of date** (cutoff).
- Build features using only data **on or before** the cutoff.
- Create labels using only data **after** the cutoff in a defined horizon.


In [ ]:
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor


REPO_ROOT = Path("..").resolve()
RAW_DIR_A = (REPO_ROOT / "data" / "raw" / "lighthouse_csv_v7").resolve()
RAW_DIR_B = (REPO_ROOT / "data" / "raw").resolve()
DATA_DIR = RAW_DIR_A if RAW_DIR_A.exists() else RAW_DIR_B

OUT_DIR = (REPO_ROOT / "output" / "ml-predictions").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Out dir:", OUT_DIR)


def require_csv(stem: str) -> pd.DataFrame:
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}.")
    return pd.read_csv(path, encoding="utf-8-sig")


def to_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.date


def to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True)


def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out


def eval_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def export_predictions_json(prediction_type: str, entity_type: str, df_out: pd.DataFrame, id_col: str, score_col: str, label_col: str | None = None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    rows = []
    for _, r in df_out.iterrows():
        rows.append(
            {
                "predictionType": prediction_type,
                "entityType": entity_type,
                "entityId": int(r[id_col]),
                "score": float(r[score_col]),
                "label": None if label_col is None else (None if pd.isna(r[label_col]) else str(r[label_col])),
                "payloadJson": json.dumps({k: v for k, v in r.items() if k not in {id_col, score_col, label_col}}, default=str),
            }
        )
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print("Wrote:", out_path, "rows=", len(rows))


In [ ]:
supporters = require_csv("supporters")
donations = require_csv("donations")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")

max_date = donations["donation_date"].max()
cutoff = max_date - pd.Timedelta(days=90)
label_end = cutoff + pd.Timedelta(days=90)
print("Max donation_date:", max_date.date())
print("Cutoff:", cutoff.date(), "Label window end:", label_end.date())

# Donations split into past vs future window for labeling
past = donations[donations["donation_date"] <= cutoff].copy()
future = donations[(donations["donation_date"] > cutoff) & (donations["donation_date"] <= label_end)].copy()

# Label: did the supporter donate in the next 90 days?
y = future.groupby("supporter_id")["donation_id"].count().rename("donated_next_90d")
y = (y > 0).astype(int).reset_index()

# Features: supporter attributes
base = supporters[["supporter_id","supporter_type","relationship_type","region","country","status","acquisition_channel","first_donation_date"]].copy()
base["first_donation_date"] = pd.to_datetime(base["first_donation_date"], errors="coerce")

# Features: donation history aggregates as-of cutoff
past_monetary = past[past["donation_type"] == "Monetary"].copy()

agg = past.groupby("supporter_id").agg(
    donation_count=("donation_id","count"),
    last_donation_date=("donation_date","max"),
    distinct_campaigns=("campaign_name", lambda s: s.dropna().nunique()),
    distinct_channels=("channel_source", lambda s: s.dropna().nunique()),
    recurring_any=("is_recurring", lambda s: int((s == True).any())),
).reset_index()

agg_m = past_monetary.groupby("supporter_id").agg(
    monetary_count=("donation_id","count"),
    monetary_sum=("amount", "sum"),
    monetary_avg=("amount", "mean"),
    monetary_max=("amount", "max"),
).reset_index()

df = base.merge(agg, on="supporter_id", how="left").merge(agg_m, on="supporter_id", how="left").merge(y, on="supporter_id", how="left")
df["donated_next_90d"] = df["donated_next_90d"].fillna(0).astype(int)

df["recency_days"] = (cutoff - df["last_donation_date"]).dt.days
df["recency_days"] = df["recency_days"].fillna(9999).clip(lower=0)

for col in ["donation_count","distinct_campaigns","distinct_channels","recurring_any","monetary_count","monetary_sum","monetary_avg","monetary_max"]:
    df[col] = df[col].fillna(0)

df = df[df["status"].isin(["Active","Inactive"])].copy()

print("Rows:", len(df), "Pos rate:", df["donated_next_90d"].mean())
df.head()


In [ ]:
# Train/test split
target = "donated_next_90d"
features = [
    "supporter_type","relationship_type","region","country","status","acquisition_channel",
    "donation_count","recency_days","distinct_campaigns","distinct_channels","recurring_any",
    "monetary_count","monetary_sum","monetary_avg","monetary_max"
]

X = df[features].copy()
y = df[target].copy()

cat_cols = ["supporter_type","relationship_type","region","country","status","acquisition_channel"]
num_cols = [c for c in features if c not in cat_cols]

pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


In [ ]:
# Predictive model (ensemble)
gb = Pipeline(steps=[
    ("pre", pre),
    ("model", GradientBoostingClassifier(random_state=42))
])
gb.fit(X_train, y_train)
proba = gb.predict_proba(X_test)[:,1]
pred = (proba >= 0.5).astype(int)
print("Predictive (GB):", eval_classification(y_test, pred, proba))

# Explanatory model (logistic regression)
lr = Pipeline(steps=[
    ("pre", pre),
    ("model", LogisticRegression(max_iter=2000))
])
lr.fit(X_train, y_train)
proba2 = lr.predict_proba(X_test)[:,1]
pred2 = (proba2 >= 0.5).astype(int)
print("Explanatory (LogReg):", eval_classification(y_test, pred2, proba2))


In [ ]:
# Score all supporters as-of cutoff (use predictive model)
df_out = df[["supporter_id"] + features].copy()
df_out["risk_score"] = gb.predict_proba(df_out[features])[:,1]
df_out["risk_band"] = pd.qcut(df_out["risk_score"], q=4, labels=["Low","Medium","High","Very High"])

export_predictions_json(
    prediction_type="donor_lapse_90d",
    entity_type="Supporter",
    df_out=df_out[["supporter_id","risk_score","risk_band","recency_days","donation_count","monetary_sum","recurring_any","acquisition_channel","supporter_type"]],
    id_col="supporter_id",
    score_col="risk_score",
    label_col="risk_band"
)


## 3) Modeling & Feature Selection (Rubric)

- Predictive model: tree/ensemble
- Explanatory model: linear/logistic regression


## 4) Evaluation & Interpretation (Rubric)

Interpret in business terms, and discuss real-world costs of errors.

## 5) Causal and Relationship Analysis (Rubric)

Discuss relationships, confounding risks, and where correlation ≠ causation.


## 6) Deployment Notes (Rubric)

Export predictions to JSON and import into the deployed app:
- `POST /api/ml/import?replace=true` (admin-only)
- View in `/app/ml` (Staff Portal → ML Insights)
